# The Magic of Transformers: A Complete and Practical Tutorial

Welcome to modern Natural Language Processing (NLP). In this tutorial, we will strip away the magic and look at the analytical pipeline of a Transformer.

### The Analytical Pipeline:
1. **Tokenization:** Human text must be mathematically encoded into integer IDs.
2. **Embedding & Attention (The Forward Pass):** The integers are converted into dense vectors, and the Self-Attention mechanism contextualizes them.
3. **Downstream Tasks:** We will apply this architecture to two practical problems: **Sequence Classification** (Sentiment Analysis via an Encoder) and **Autoregressive Generation** (Text Generation via a Decoder).

In [ ]:
# Cell 2: Environment Setup and Imports
# In a real environment, you must install: !pip install -q transformers torch

import torch
from transformers import AutoTokenizer, AutoModel, pipeline, AutoModelForCausalLM

# Set random seed for analytical reproducibility
torch.manual_seed(42)

print(f"PyTorch Version: {torch.__version__}")
print("Hugging Face Transformers library loaded successfully.")

## Step 1: Tokenization (Text to Tensors)

Neural networks cannot process strings; they require numbers. The **Tokenizer** is responsible for chopping a string into smaller pieces (sub-words) and mapping them to a massive vocabulary dictionary.

It also generates an **Attention Mask**. Because neural networks process data in fixed-size batches, shorter sentences are padded with zeros. The attention mask tells the Self-Attention mechanism to mathematically ignore (mask out) those padding tokens so they don't corrupt the context.

In [ ]:
# Cell 4: Analyzing the Tokenizer

# We will use the tokenizer from BERT (Bidirectional Encoder Representations from Transformers)
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

sample_text = "Transformers are incredibly analytical machines."

# Convert text to PyTorch tensors (return_tensors="pt")
inputs = tokenizer(sample_text, return_tensors="pt")

print(f"Original Text: {sample_text}\n")
print(f"Input IDs (The mathematical representation):\n{inputs['input_ids']}\n")
print(f"Attention Mask (1 = pay attention, 0 = ignore padding):\n{inputs['attention_mask']}\n")

# Analytically reverse the process to see the sub-word chunking
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(f"Sub-word Tokens: {tokens}")
print("Analytical Note: Notice how 'incredibly' and 'analytical' might be split into sub-words depending on the vocabulary, and special [CLS] and [SEP] tokens are added to mark the sequence boundaries.")

## Step 2: The Forward Pass and Hidden States

What exactly happens inside the Transformer? When we pass our `input_ids` through the model, they are projected into high-dimensional space. 

The output of a base Transformer is called the **Hidden State**. This is a 3D tensor of shape: 
$$Batch Size \times Sequence Length \times Hidden Dimension$$

This tensor is the "contextualized mathematical understanding" of the sentence. Every single token vector now contains information about every *other* token vector in the sentence, thanks to the Self-Attention mechanism.

In [ ]:
# Cell 6: Extracting the Hidden State

# Load the base BERT model (no classification head, just the core architecture)
model = AutoModel.from_pretrained(model_checkpoint)

# Perform the forward pass without tracking gradients (saves memory)
with torch.no_grad():
    outputs = model(**inputs)

# Extract the last hidden state
last_hidden_state = outputs.last_hidden_state

print("--- The Mathematical Output of the Transformer ---")
print(f"Hidden State Tensor Shape: {last_hidden_state.shape}")

batch_size = last_hidden_state.shape[0]
sequence_length = last_hidden_state.shape[1]
hidden_dim = last_hidden_state.shape[2]

print(f"Batch Size: {batch_size} (We processed 1 sentence)")
print(f"Sequence Length: {sequence_length} (There are {sequence_length} tokens in our sentence)")
print(f"Hidden Dimension: {hidden_dim} (Each token is represented by an array of {hidden_dim} floating-point numbers)")

## Step 3: Practical Application 1 - Sequence Classification

Now that we understand the base architecture, how do we use it? 

By adding a simple Dense (Linear) layer on top of the Transformer's Hidden State, we can train it to classify the sentence. Hugging Face makes this incredibly easy using the `pipeline` API, which abstracts the tokenization, forward pass, and output decoding into a single step.

We will use a fine-tuned model for **Sentiment Analysis**.

In [ ]:
# Cell 8: Sentiment Analysis Pipeline

# Load a pipeline specifically built for text classification
# By default, this uses a distilled version of BERT fine-tuned on sentiment data
sentiment_analyzer = pipeline("sentiment-analysis")

reviews = [
    "The data processing was highly efficient and the math was sound.",
    "The model completely failed to converge, wasting hours of GPU time."
]

print("--- Sentiment Analysis Results ---")
for review in reviews:
    result = sentiment_analyzer(review)[0]
    print(f"Text: '{review}'")
    print(f"Prediction: {result['label']} | Confidence Score: {result['score'] * 100:.2f}%\n")

## Step 4: Practical Application 2 - Autoregressive Generation

BERT is an **Encoder-only** model; it is great at understanding text. To *generate* text (like ChatGPT), we need a **Decoder** model, like GPT-2.

Decoders are "Autoregressive." They take a sequence of tokens, predict the *next* most likely token, append it to the sequence, and feed the whole thing back into themselves in a continuous loop.

In [ ]:
# Cell 10: Text Generation with GPT-2

# We load a model specifically designed for Causal Language Modeling (predicting the next word)
generation_model_id = "gpt2"
gen_tokenizer = AutoTokenizer.from_pretrained(generation_model_id)
gen_model = AutoModelForCausalLM.from_pretrained(generation_model_id)

prompt = "The future of Artificial Intelligence relies heavily on"
gen_inputs = gen_tokenizer(prompt, return_tensors="pt")

print(f"Input Prompt: '{prompt}'")
print("Generating continuation...\n")

# Execute the autoregressive loop
with torch.no_grad():
    output_sequences = gen_model.generate(
        input_ids=gen_inputs['input_ids'],
        attention_mask=gen_inputs['attention_mask'],
        max_length=30,               # Maximum number of tokens to output
        num_return_sequences=1,      # How many different variations to generate
        no_repeat_ngram_size=2,      # Prevents the model from repeating the same phrases
        do_sample=True,              # Allows for creative randomness
        temperature=0.7              # Flattens the probability curve slightly
    )

# Decode the mathematical output back into human text
generated_text = gen_tokenizer.decode(output_sequences[0], skip_special_tokens=True)

print("--- Generated Output ---")
print(generated_text)

## Conclusion

You have successfully navigated the architecture that powers modern AI!

**Key Analytical Takeaways:**
1. **Self-Attention over Recurrence:** Transformers do not process sequentially. They process the entire sequence at once, relying on Attention Matrices and Positional Encodings to understand order and context. This allows them to utilize highly parallelized GPU computation.
2. **The Base is Universal:** Whether you are translating languages, analyzing sentiment, or generating text, the base mechanism—converting tokens to $H$-dimensional contextualized embeddings—remains exactly the same.
3. **Encoders vs. Decoders:** Use Encoders (BERT) when you need deep, bidirectional understanding of a complete text (Classification, Named Entity Recognition). Use Decoders (GPT) when you need to predict the future (Text Generation).